<a href="https://colab.research.google.com/github/NismaK/ML_Homeworkbook/blob/main/ML_zoomcamp_homework_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Zoomcamp 2026 — Homework 3


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.linear_model import LogisticRegression

# Load the dataset directly from the official course URL
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'
df = pd.read_csv(url)
print('Dataset shape:', df.shape)
df.head()

Dataset shape: (5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation
Categorical missing values are replaced with `'NA'`; numerical feature missing values are replaced with `0.0`. The target `converted` is retained separately for splitting and removed from the feature data.

In [2]:
df = df.copy()
categorical_features = df.select_dtypes(include='object').columns.tolist()
numerical_features = [c for c in df.select_dtypes(exclude='object').columns if c != 'converted']

for col in categorical_features:
    df[col] = df[col].fillna('NA')
for col in numerical_features:
    df[col] = df[col].fillna(0.0)

print('Missing values after preparation:')
print(df.isna().sum())
print('\nCategorical features:', categorical_features)
print('Numerical features:', numerical_features)

Missing values after preparation:
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

Categorical features: ['lead_source', 'industry', 'employment_status', 'location']
Numerical features: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


## Question 1 — Most frequent `industry`

In [3]:
industry_mode = df['industry'].mode()[0]
print('Q1 — Mode of industry:', industry_mode)

Q1 — Mode of industry: technology


## Question 2 — Correlation between numerical features


In [4]:
corr = df[numerical_features].corr()
candidate_pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
pair_correlations = {pair: corr.loc[pair[0], pair[1]] for pair in candidate_pairs}
for pair, value in pair_correlations.items():
    print(f'{pair[0]} and {pair[1]}: {value:.4f}')
q2_answer = max(pair_correlations, key=lambda pair: pair_correlations[pair])
print('\nQ2 — Pair with the biggest correlation:', q2_answer)

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228

Q2 — Pair with the biggest correlation: ('interaction_count', 'lead_score')


## Split the data
These are the exact split calls requested in the homework. `converted` is separated from the feature data.

In [5]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted']
y_val = df_val['converted']
X_train = df_train.drop(columns=['converted'])
X_val = df_val.drop(columns=['converted'])

print('Train:', X_train.shape, 'Validation:', X_val.shape, 'Test:', df_test.drop(columns=['converted']).shape)
assert 'converted' not in X_train.columns
assert 'converted' not in X_val.columns

Train: (3000, 8) Validation: (1000, 8) Test: (1000, 8)


## Question 3 — Mutual information


In [6]:
mi_scores = {}
for col in categorical_features:
    score = mutual_info_score(y_train, X_train[col])
    mi_scores[col] = round(score, 2)

print('Mutual information scores:', mi_scores)
q3_answer = max(mi_scores, key=mi_scores.get)
print('Q3 — Biggest mutual information score:', q3_answer)

Mutual information scores: {'lead_source': np.float64(0.03), 'industry': np.float64(0.0), 'employment_status': np.float64(0.02), 'location': np.float64(0.0)}
Q3 — Biggest mutual information score: lead_source


## Question 4 — Logistic regression accuracy


In [7]:
dv = DictVectorizer(sparse=False)
X_train_dict = X_train.to_dict(orient='records')
X_val_dict = X_val.to_dict(orient='records')
X_train_encoded = dv.fit_transform(X_train_dict)
X_val_encoded = dv.transform(X_val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_encoded, y_train)
y_pred = model.predict(X_val_encoded)
original_accuracy = accuracy_score(y_val, y_pred)
print(f'Q4 — Validation accuracy: {original_accuracy:.2f}')

Q4 — Validation accuracy: 0.65


## Question 5 — Feature elimination


In [8]:
elimination_results = {}
for feature in X_train.columns:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced_encoded = dv_reduced.fit_transform(X_train_reduced.to_dict(orient='records'))
    X_val_reduced_encoded = dv_reduced.transform(X_val_reduced.to_dict(orient='records'))

    reduced_model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    reduced_model.fit(X_train_reduced_encoded, y_train)
    reduced_accuracy = accuracy_score(y_val, reduced_model.predict(X_val_reduced_encoded))
    elimination_results[feature] = {
        'accuracy_without_feature': reduced_accuracy,
        'difference': original_accuracy - reduced_accuracy,
    }

elimination_df = pd.DataFrame(elimination_results).T
display(elimination_df)

q5_candidates = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
q5_answer = min(q5_candidates, key=lambda feature: elimination_results[feature]['difference'])
print('Q5 — Smallest difference among the listed options:', q5_answer)

,accuracy_without_feature,difference
lead_source,0.642,0.003
industry,0.645,0.000
employment_status,0.645,0.000
location,0.645,0.000
annual_income,0.724,-0.079
number_of_courses_viewed,0.643,0.002
interaction_count,0.601,0.044
lead_score,0.644,0.001


Q5 — Smallest difference among the listed options: number_of_courses_viewed


## Question 6 — Regularized logistic regression


In [9]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
C_accuracies = {}

for C in C_values:
    regularized_model = LogisticRegression(
        solver='liblinear', C=C, max_iter=1000, random_state=42
    )
    regularized_model.fit(X_train_encoded, y_train)
    accuracy = accuracy_score(y_val, regularized_model.predict(X_val_encoded))
    C_accuracies[C] = accuracy
    print(f'C={C}: accuracy={accuracy:.3f}')

best_accuracy = max(C_accuracies.values())
q6_answer = min(C for C, accuracy in C_accuracies.items() if accuracy == best_accuracy)
print('Q6 — Best C (smallest C in case of a tie):', q6_answer)

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
Q6 — Best C (smallest C in case of a tie): 0.001


## Final answers


In [10]:
print('Q1:', industry_mode)
print('Q2:', ' and '.join(q2_answer))
print('Q3:', q3_answer)
print(f'Q4: {original_accuracy:.2f}')
print('Q5:', q5_answer)
print('Q6:', q6_answer)

Q1: technology
Q2: interaction_count and lead_score
Q3: lead_source
Q4: 0.65
Q5: number_of_courses_viewed
Q6: 0.001
